# 13 — Baselines & ablations

Variant runs to isolate each component's contribution: BM25-only, dense-only, RRF (no rerank), full pipeline. Writes a side-by-side table to `_artifacts/13_baselines/ablations.json`.

In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'apps').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')
from eval.search_eval import run_search_eval
from apps.backend.graph.neo4j_client import get_driver

driver = get_driver()
variants = ['bm25_only', 'dense_only', 'rrf_no_rerank', 'full']
rows = {}
for v in variants:
    try:
        rows[v] = run_search_eval(driver, REPO_ROOT / 'data' / 'bench' / 'queries.jsonl', k=10, variant=v)
    except TypeError:
        rows[v] = {'note': 'variant kwarg not yet wired; default run only'}
        break

ART = REPO_ROOT / 'notebooks' / '_artifacts' / '13_baselines'
ART.mkdir(parents=True, exist_ok=True)
(ART / 'ablations.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2, default=str))
print('artifact:', ART / 'ablations.json')
